# A1 Flight Route Planner — demo

**Does informed search actually pay for itself on a real network?**

This notebook is the demo for CMPE 180A project A1. It runs the delivered
system end to end on the world airline network — 3,387 airports and 66,332
routes — and shows the three things the project claims:

1. Three query modes, from one interface, on one pinned dataset
2. A\* returns **exactly** Dijkstra's answer while expanding far fewer airports
3. The answer is checked against an independent implementation, not just
   against our own tests

## Two rules this notebook follows

**Every measured number is read from a committed `results.json`, never
recomputed here.** A demo that re-runs a timing loop on stage is a demo that
waits, and a number produced live cannot be compared against the report. The
experiments that produced them are in `experiments/`, and
`tests/experiments/` re-derives their answers on every test run.

**The data is pinned and verified.** Opening a snapshot re-hashes every file
against its manifest and refuses to proceed on a mismatch — so a demo that
would have shown the wrong data fails loudly instead of quietly.

What is *computed* live below is only what is fast and deterministic: the
route queries themselves, and the boundary cases.

> **Nothing here is stored output.** The notebook is committed with empty
> cells on purpose; run it top to bottom.

## Setup

Only this first cell differs between Colab and a local checkout. Locally,
`uv sync --group notebooks` (or `make notebook`) has already installed
everything.

In [ ]:
# In Colab, clone the repository and install the package plus the two
# mapping libraries:
#   !git clone https://github.com/dgwartney/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima
#   %pip install -q folium pyproj
# Then open this notebook from the clone.
import json
import sys
from pathlib import Path

try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit("install the package first -- see the comment above") from error

# This notebook lives in `notebooks/`, so the repository root is one up. The
# demo modules are not part of the wheel: they sit outside it deliberately, so
# that running them exercises the package's public re-exports the same way a
# user would.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src" / "demos"))

EXPERIMENTS = REPO / "experiments"


def recorded(slug):
    """Return one experiment's committed answer."""
    return json.loads((EXPERIMENTS / slug / "results.json").read_text())


print(f"repository: {REPO}")

## 1. The data, and why it is pinned

`data/processed/` is build output — `make flight_network` overwrites it and the
filenames do not change. A snapshot is an immutable, content-addressed copy:
its id is a hash of its contents, so it can be superseded but never edited.

`Snapshot.open()` re-hashes every file before returning a row.

In [ ]:
from flight_planner.experiments import Snapshot
from route_query_example import SNAPSHOTS_DIR, SNAPSHOT_ID

# Opening it is the verification: every file is re-hashed against the
# manifest before a row is returned.
snapshot = Snapshot.open(SNAPSHOTS_DIR / SNAPSHOT_ID)
catalog = snapshot.catalog()

print(f"snapshot   {snapshot.snapshot_id}")
print(f"criteria   {snapshot.criteria or '(none -- the whole world)'}")
print(f"airports   {len(catalog.airports):,}")
print(f"routes     {len(catalog.routes):,}")
print()
print("Every file above was re-hashed against the manifest on the way in.")

## Watch it work, before any table

One map, right now: the whole 66,332-route network as a backdrop you can
switch on with the layer control; `HNL ↔ BDL` under all three query modes;
and `SYD ↔ JFK`, a route that leaves Sydney in opposite directions depending
on which algorithm answers it. Toggle layers, watch A* fly its answer, then
scroll down for the numbers behind it.

In [ ]:
import re

from IPython.display import HTML, display

# Borrow the deck's own KPI visual language for headline numbers, read from
# the file rather than hand-copied so it can never drift from the deck's
# colours. Styles are inlined on each span rather than injected as a shared
# <style> block: GitHub, Colab and nbviewer all strip <style>/<script> tags
# from notebook HTML output, which would silently drop class-based styling
# while leaving inline `style="..."` attributes intact.
deck_css = (REPO / "docs" / "templates" / "deck.css").read_text()
root_block = re.search(r":root\s*\{([^}]*)\}", deck_css, re.S)
assert root_block, "deck.css structure changed -- update this regex"
root_vars = dict(re.findall(r"--([\w-]+):\s*([^;]+);", root_block.group(1)))

# Note: the deck's --astar/--dim are its dark-surface hex values. The
# light-mode PNG charts below use slightly different shades of the same
# colours -- a cosmetic mismatch this notebook accepts rather than building
# a second CSS variant just for inline KPIs.
_KPI_VALUE_STYLE = (
    f"color:{root_vars['astar']};font-weight:800;font-size:2em;"
    "display:block;line-height:1.1;margin-top:0.2em;"
)
_KPI_LABEL_STYLE = (
    f"color:{root_vars['dim']};font-size:0.62em;display:block;"
    "margin-bottom:0.9em;"
)


def kpi(value, label):
    """A headline stat callout, styled like the deck's `.kpi`/`.kpi-label`."""
    return HTML(
        f'<span style="{_KPI_VALUE_STYLE}">{value}</span>'
        f'<span style="{_KPI_LABEL_STYLE}">{label}</span>'
    )


In [ ]:
from flight_planner import BFS, Dijkstra
from flight_planner.viz import Palette, PathLayer, RouteMap
import folium
import folium.plugins as plugins

from route_query_example import compare_modes, format_result

planner = catalog.planner()
ORIGIN, DESTINATION = "HNL", "BDL"
results = compare_modes(planner, ORIGIN, DESTINATION)

In [ ]:
palette = Palette()
hero_map = RouteMap(palette=palette)
hero_map.routes(catalog.routes)  # the whole network, a toggle-on backdrop
hero_map.airports([planner.iata_lookup[code] for code in ("HNL", "BDL", "SYD", "JFK")])

# Palette.series() keys colour by algorithm name alone, so HNL-BDL's "BFS"
# layer and SYD-JFK's "BFS" layer share a hue (same for "Dijkstra"). The two
# pairs are told apart in the layer control, not by colour: PathLayer.name()
# leads each label with the pair it answers, which also groups the three
# HNL-BDL layers together above the two SYD-JFK ones.
hero_map.result(results["fewest stops"], name="BFS")
hero_map.result(results["shortest distance"], name="Dijkstra")

# Built explicitly, not through .result(), so the layer stays addressable
# below for the animated overlay.
astar_layer = PathLayer.from_result(
    results["shortest distance (A*)"], palette=palette, name="A*"
)
hero_map.add(astar_layer)

for label, algorithm in {"Dijkstra": Dijkstra(), "BFS": BFS()}.items():
    hero_map.result(planner.search_route("SYD", "JFK", algorithm), name=label)

# Everything below goes on through finish(decorate=...), not onto the map it
# returns. The layer control is attached last and collects its overlays at
# render time, so a feature group added afterwards is listed by a control that
# runs before the variable holding it exists -- Leaflet throws, and the map
# loses its whole legend without a single Python error. See RouteMap.finish().
def add_the_flourishes(folium_map):
    """Draw the animated A* path and the two map controls."""
    # The "watch it fly" flourish: the same A* path, animated. drawn_points()
    # is populated by now, which is why this runs here and not earlier.
    antpath_group = folium.FeatureGroup(
        name=f"{ORIGIN}-{DESTINATION} · A* (animated)", show=True
    )
    plugins.AntPath(
        locations=astar_layer.drawn_points(),
        color=palette.series("A*"),
        weight=4,
        delay=800,
        dash_array=[12, 24],
    ).add_to(antpath_group)
    antpath_group.add_to(folium_map)

    plugins.Fullscreen(position="topleft").add_to(folium_map)
    plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(folium_map)


# Capture the built map explicitly -- hero_map's own _repr_html_ calls
# finish() again on render and would silently rebuild a plugin-less map.
folium_map = hero_map.finish(decorate=add_the_flourishes)
folium_map

This one cell needs **network access**: Leaflet tiles from OpenStreetMap,
plus three small third-party scripts pulled from CDNs for the extras above --
`leaflet-ant-path` (jsdelivr) for the animated route, and `leaflet.fullscreen`
(jsdelivr) and `leaflet-minimap` (cdnjs) for the two map controls. None of
that is the finding: the animation and the controls are chrome. If the room
has no network, the three routes drawn above are committed as PNGs under
`slides/images/` and appear on the deck's route-map slide, so the finding
survives without the live render.

### The network itself

Before narrowing to one pair: what does 3,387 airports and 66,332 routes
actually look like? `size`, `components` and `top_hubs` below are read from
`experiments/graph-stats/`, the same way the search-cost numbers further down
are -- recorded, not recomputed.

In [ ]:
graph_stats = recorded("graph-stats")
assert graph_stats["snapshot"]["id"] == SNAPSHOT_ID, "measured on other data"

size = graph_stats["results"]["size"]
components = graph_stats["results"]["components"]

print(f"{size['airports']:,} airports, {size['routes']:,} routes, "
      f"{size['airport_pairs']:,} distinct pairs "
      f"({size['parallel_routes']:,} routes run parallel to another on the same pair)")
print(f"density {size['density']:.4%} of a complete graph")
print(f"{components['weak_count']} weakly-connected components; the largest "
      f"holds {components['largest_weak']:,} of {size['airports']:,} airports")

In [ ]:
import importlib.util
import tempfile

from IPython.display import Image

CHARTS = Path(tempfile.mkdtemp())


def _load_plots(experiment_slug, module_name):
    """Load an experiment's plots.py without colliding with another one.

    Every experiment's chart module is named `plots.py`, so a bare
    `import plots` after `sys.path.insert` would just return whichever one
    was imported first -- `sys.modules` caches by module name, not by path.
    Loading directly from the file sidesteps that collision entirely.
    """
    spec = importlib.util.spec_from_file_location(
        module_name, EXPERIMENTS / experiment_slug / "plots.py"
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


graph_plots = _load_plots("graph-stats", "graph_stats_plots")

display(Image(graph_plots.top_hubs(
    graph_stats["results"]["top_hubs"], CHARTS / "top-hubs.png", mode="light")))

busiest = graph_stats["results"]["top_hubs"][0]
display(kpi(
    f'{busiest["out_degree"]:,}',
    f'routes out of {busiest["iata_code"]}, the busiest airport in the network',
))


In [ ]:
# The out-degree histogram isn't in graph-stats' recorded numbers -- only its
# bucketed digest is -- so it's computed here instead, straight from the
# catalog already opened and hash-verified above. That's a structural fact
# about pinned data, not a measured claim, so it falls under this notebook's
# "fast and deterministic" rule rather than its "read, never recomputed" one.
from collections import Counter

out_degree = Counter(route.origin.iata_code for route in catalog.routes)
out_counts = Counter(out_degree[airport.iata_code] for airport in catalog.airports)

display(Image(graph_plots.degree_distribution(
    out_counts, CHARTS / "degree-distribution.png", mode="light")))

## 2. One question, three different right answers

`FlightPlanner` is handed the algorithm rather than containing one, so the
same call answers three different questions. `HNL → BDL` (Honolulu to
Hartford) is the clearest case.

Watch the **unit** column. BFS's cost is a hop count; the other two are
kilometres. They are never summed, and "BFS won" never means "BFS found a
shorter route".

In [ ]:
print(f"{ORIGIN} -> {DESTINATION}\n")
print(f"{'mode':<24}{'cost':>20} {'unit':<5} {'expanded':>14}  route")
for name, result in results.items():
    print(f"{name:<24}{format_result(ORIGIN, result)}")

Three things to read off that, and they are the whole project in miniature.

**BFS disagrees because it was asked something else.** Its two-leg itinerary
is 8,616.2 km against Dijkstra's 8,071.5 — so skipping one stop costs 544.7 km.

**A\* agrees with Dijkstra exactly**, not approximately. The assertion below
is on the raw floats, not a rounded display.

**A\* paid about 100× less** to reach the same answer.

In [ ]:
dijkstra = results["shortest distance"]
astar = results["shortest distance (A*)"]

assert astar.cost == dijkstra.cost, "A* and Dijkstra must agree exactly"
assert [leg.flight_number for leg in astar.path] == [
    leg.flight_number for leg in dijkstra.path
], "and on the same itinerary"

print(f"identical to the last bit: {dijkstra.cost!r} == {astar.cost!r}")
ratio = dijkstra.nodes_expanded / astar.nodes_expanded
print(f"expansions: Dijkstra {dijkstra.nodes_expanded}, "
      f"A* {astar.nodes_expanded} ({ratio:.0f}x fewer)")

display(kpi(
    f'{ratio:.0f}×',
    f'fewer airports expanded by A* than Dijkstra for the same answer, {ORIGIN}→{DESTINATION}',
))


## 3. The measured claim

These are **read, not recomputed** — `experiments/search-cost/` produced them
on the same snapshot, and its `results.json` is committed beside the notebook
that wrote it.

In [ ]:
search_cost = recorded("search-cost")
assert search_cost["snapshot"]["id"] == SNAPSHOT_ID, "measured on other data"

print(f"{'query':<10}{'BFS':>8}{'Dijkstra':>10}{'A*':>6}{'A* saving':>12}"
      f"{'Dijkstra km':>15}{'A* km':>15}")
for row, agree in zip(search_cost["results"]["comparison"],
                      search_cost["results"]["agreement"]):
    expanded = row["expanded"]
    print(f"{row['pair']:<10}{expanded['BFS']:>8,}{expanded['Dijkstra']:>10,}"
          f"{expanded['A*']:>6,}{agree['expansion_ratio']:>11,.0f}x"
          f"{agree['dijkstra_km']:>15,.3f}{agree['astar_km']:>15,.3f}")

gaps = {abs(a["dijkstra_km"] - a["astar_km"])
        for a in search_cost["results"]["agreement"]}
print(f"\nlargest A* vs Dijkstra difference across all five: {max(gaps)} km")

display(kpi(
    f'{max(gaps):.3f} km',
    'largest A* vs Dijkstra distance difference across all five measured pairs',
))


In [ ]:
# The charts are drawn live, but from the *recorded* numbers -- `plots.py` is
# the same module `experiments/search-cost` used to render the report's
# figures, so these are the report's figures, not a lookalike.
from IPython.display import Image, display

sys.path.insert(0, str(EXPERIMENTS / "search-cost"))
import plots  # noqa: E402

In [ ]:
display(Image(plots.nodes_expanded(
    search_cost["results"]["comparison"],
    CHARTS / "nodes-expanded.png",
    mode="light",
)))

**130× to 784× fewer expansions, and the distances match to the last decimal
place.** That is the project's central claim, measured rather than asserted.

Runtime and the fitted growth exponents come from the same file. The absolute
milliseconds are a property of the machine that recorded them; the exponents
and the node counts are properties of the algorithms and reproduce anywhere.

In [ ]:
environment = search_cost["results"]["environment"]
print(f"recorded on {environment['cpu']}, {environment['cores']} cores, "
      f"Python {environment['python']}\n")

print(f"{'narrowing':<18}{'V + E':>8}{'BFS ms':>9}{'Dijkstra ms':>13}"
      f"{'A* ms':>8}{'A* vs Dij':>11}")
for row in search_cost["results"]["runtime_series"]:
    ms = row["median_ms"]
    print(f"{row['label']:<18}{row['size']:>8,}{ms['BFS']:>9.3f}"
          f"{ms['Dijkstra']:>13.3f}{ms['A*']:>8.3f}"
          f"{ms['Dijkstra'] / ms['A*']:>10.1f}x")

print()
for series, fit in search_cost["results"]["scaling"].items():
    print(f"{series:<12} growth exponent {fit['exponent']:.2f}  "
          f"(r^2 {fit['r_squared']:.3f})")

In [ ]:
# Log-log, so a power law reads as a straight line whose slope is the growth
# exponent above. Watch Dijkstra dip between 11,315 and 51,536 V + E: a graph
# 4.6x larger, answered 1.4x faster.
display(Image(plots.runtime_vs_size(
    search_cost["results"]["runtime_series"],
    CHARTS / "runtime.png",
    mode="light",
)))

Only graph construction reaches its O(V + E) exponent. Every *search* comes in
well below its bound, and the fits get worse as the algorithm gets smarter —
because a worst-case bound describes a search that exhausts the graph, and
none of these do. They stop on arrival.

## 4. Where a table stops being enough

Three pairs, from `experiments/route-map/`. The third is the one no column in
the table above can explain.

In [ ]:
route_map_results = recorded("route-map")

print(f"{'pair':<10}{'BFS legs':>10}{'BFS km':>12}{'Dij legs':>10}"
      f"{'Dij km':>12}{'penalty km':>13}")
for pair, row in route_map_results["results"]["comparison"].items():
    bfs, dij = row["BFS"], row["Dijkstra"]
    print(f"{pair:<10}{bfs['legs']:>10}{bfs['km']:>12,.1f}"
          f"{dij['legs']:>10}{dij['km']:>12,.1f}{row['km_penalty']:>13,.1f}")

`SYD-JFK` is the interesting row: **the same two legs either way, 7,057 km
apart.** Hop count cannot explain that, and neither can any column above. The
two answers leave Sydney in opposite directions -- see the hero map near the
top of this notebook, where both `SYD-JFK` legs are drawn and toggleable.

## 5. How we know the answers are right

Unit tests show the code does what its author expected. Only an **independent
implementation** can show the expectation was right — so every algorithm is
wrapped behind the same interface and run against NetworkX on the same graph.

In [ ]:
parity = recorded("networkx-parity")
print(f"oracle: {parity['results']['oracle']}")
print(f"total cost mismatches: {parity['results']['total_cost_mismatches']}\n")

print(f"{'algorithm':<12}{'pairs':>8}{'cost mismatches':>18}"
      f"{'worst divergence km':>22}{'identical paths':>18}")
for name, row in parity["results"]["parity"].items():
    print(f"{name:<12}{row['pairs_checked']:>8}{row['cost_mismatches']:>18}"
          f"{row['largest_absolute_divergence_km']:>22.2e}"
          f"{row['identical_paths']:>13}/{row['pairs_checked']}")

total_pairs = sum(r["pairs_checked"] for r in parity["results"]["parity"].values())
display(kpi(
    parity["results"]["total_cost_mismatches"],
    f'cost mismatches against the independent NetworkX oracle, across {total_pairs:,} queries',
))


The worst divergence is 3.6e-12 km — float summation order, not disagreement.

BFS matching on only 142 of 200 paths is a *finding, not a failure*: when
several routes tie at the same hop count, "fewest stops" does not name one of
them, so two correct implementations may return different ties. Their costs
agree on all 200.

### The boundary cases, computed live

These are cheap, so there is no reason to read them from a file.

In [ ]:
from flight_planner.errors import FlightPlannerError

print("unreachable destination :", planner.find_shortest_route("SPI", "JFK"))
print("origin == destination   :", planner.find_shortest_route("SFO", "SFO"))
try:
    planner.find_shortest_route("ZZZ", "SFO")
except FlightPlannerError as error:
    print(f"unknown airport code    : {type(error).__name__}: {error}")

The third one is the one that matters. An unknown code raises rather than
returning `(inf, [])`, because **"there is no such airport" and "there is no
such route" are different facts** — collapsing them would let a typo look like
a routing result.

The network also supplies its own cyclic and self-loop cases, so those are
tested on real data rather than only on fixtures: `PKN` has a 0.0 km self-loop
sitting beside its genuine departures.

In [ ]:
self_loops = [route for route in catalog.routes
              if route.origin == route.destination]
print(f"self-loops in the delivered network: {len(self_loops)}")
for route in self_loops:
    print(f"  {route.flight_number}  {route.origin.iata_code}"
          f"->{route.destination.iata_code}  {route.distance_km} km")

# Relaxation is strictly-less-than, so arriving again at 0 + 0 is not an
# improvement and the edge is never traversed.
through = planner.search_route("PKN", "SIN", Dijkstra())
print(f"\nPKN -> SIN  {through.cost:,.1f} km in {len(through.path)} legs, "
      f"{through.nodes_expanded} expanded")

## 6. What this does not show

Stated plainly, because a demo that only shows what worked is not evidence.

- **"Cheapest" means shortest distance, not lowest fare.** OpenFlights carries
  no fare data. Nothing here prices a ticket.
- **No schedules, so no connections.** A route is an edge whether or not the
  two flights either side of a stop can actually be connected.
- **Worst-case behaviour is not benchmarked.** Every query above succeeds and
  terminates early; an unreachable destination is the expensive case.
- **One graph shape.** The airline network is small-world — dense hubs, short
  paths. A sparse or grid-like graph would put the heuristic under real
  pressure and these exponents would not carry over.
- **The oracle is independent, not infallible.** Agreement with NetworkX means
  two implementations made the same decisions.

## Where everything lives

| | |
|---|---|
| The report | `make report` → `build/pdf/report.pdf` |
| The deck | `make deck` → generated from the report's own chapters |
| The algorithms | `src/flight_planner/{core,adt,pathfinding,geo}/` — no `heapq`, no `networkx` |
| The measurements | `experiments/*/results.json`, re-derived by `tests/experiments/` |
| This notebook's spine | `src/demos/route_query_example.py` |